In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from researchcodes import (
    define_column_desc, 
    iter_multi_csv_chunks, 
    write_std_h5, 
    count_lines_fast,
)

from tqdm.notebook import tqdm

In [2]:
# Download full Smith Extended Northern and Equatorial
# https://www-star.fnal.gov/NorthEqExtension_ugriz/index.html

# Process the text catalog

The magnitude errors need calculation, so it's better to process ahead of reading and writing.

I also need to drop rows marked by "*" and replace -100 and 0 with `np.nan` as missing values

In [3]:
# list of all the raw data file
raw_file_list = list(
    Path("usno40stds.clean.v3").glob("*v3")
)
raw_file_list

[PosixPath('usno40stds.clean.v3/109_b.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/BD+25o4655.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/Ross49.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/114_b.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/Feige34.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/Hilt760.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/BD+26o2606.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/95_b.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/G163_50-51.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/PG1528+062B.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/113_b.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/G3-33.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/GCRV9438.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/LHS-1858.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/101_c.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/114_c.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/112_b.txt.clean.v3'),
 PosixPath('usno40stds.clean.v3/107_b.txt.clean.v3'

In [4]:
# create a folder to save the processed data file

process_file_dir = Path("Northern_ugriz_processed")
process_file_dir.mkdir(exist_ok=True)

In [5]:
all_column_names = [
    "id_name",
    "ra",
    "dec",
    "SDSS_up", "SDSS_up_rms_err", "SDSS_up_nobs",
    "SDSS_gp", "SDSS_gp_rms_err", "SDSS_gp_nobs",
    "SDSS_rp", "SDSS_rp_rms_err", "SDSS_rp_nobs",
    "SDSS_ip", "SDSS_ip_rms_err", "SDSS_ip_nobs",
    "SDSS_zp", "SDSS_zp_rms_err", "SDSS_zp_nobs",
]

pbar_files = tqdm(
    raw_file_list,
    desc="Files",
    total=len(raw_file_list),
    position=0,
    leave=True,
)

for file in pbar_files:

    pbar_files.set_postfix_str(file.name, refresh=True)

    field_name = file.stem.split(".")[0]

    raw_text_catalog = pd.read_csv(
        file,
        sep=r"\s+",
        engine="python",
        header=0,
        names=all_column_names,
    )

    # the columns need to check missing values
    missing_magnitude_columns = [
        "SDSS_up", "SDSS_up_rms_err",
        "SDSS_gp", "SDSS_gp_rms_err",
        "SDSS_rp", "SDSS_rp_rms_err",
        "SDSS_ip", "SDSS_ip_rms_err",
        "SDSS_zp", "SDSS_zp_rms_err",
    ]
    missing_nobs_columns = ["SDSS_up_nobs","SDSS_gp_nobs","SDSS_rp_nobs","SDSS_ip_nobs","SDSS_zp_nobs"]
    
    # replace all missing values (-100) by np.nan
    raw_text_catalog[missing_magnitude_columns] = raw_text_catalog[missing_magnitude_columns].replace(-100, np.nan)
    # replace all missing values (0) by np.nan
    raw_text_catalog[missing_nobs_columns] = raw_text_catalog[missing_nobs_columns].replace(0, np.nan)  

    # calculate the ra ande dec errors
    # this formula (rm/sqrt(Ntot)) is 
    # in the comments of the catalog file
    for b in ["up","gp","rp","ip","zp"]:
        raw_text_catalog[f"SDSS_{b}_err"] = raw_text_catalog[f"SDSS_{b}_rms_err"]/np.sqrt(raw_text_catalog[f"SDSS_{b}_nobs"])

    # drop the columns we don't want in the final output file
    process_text_catalog = raw_text_catalog.drop(
        columns=[
            "SDSS_up_rms_err", "SDSS_up_nobs",
            "SDSS_gp_rms_err", "SDSS_gp_nobs",
            "SDSS_rp_rms_err", "SDSS_rp_nobs",
            "SDSS_ip_rms_err", "SDSS_ip_nobs",
            "SDSS_zp_rms_err", "SDSS_zp_nobs",
        ]
    )


    # Adding the field name to the standard stars to make it unique
    new_name = [field_name + f"_{id_name}" for id_name in process_text_catalog["id_name"]]
    process_text_catalog["id_name"] = new_name
    
    process_text_catalog.to_csv(
        process_file_dir / f"{field_name}_processed.csv",
        sep=",",
        index=False,
    )

Files:   0%|          | 0/110 [00:00<?, ?it/s]

# Define the HFD5 column description

In [6]:
# Define magnitude column names

# Note this magnitude column order is not necessarily
# to be the same as the column order in the text file. 

# It is OK as long as the filter names in 
# `magnitude_column_names` matches `colnames` defined 
# when reading the text file

magnitude_column_names = [
     "SDSS_up","SDSS_gp","SDSS_rp","SDSS_ip","SDSS_zp",
    "SDSS_up_err","SDSS_gp_err","SDSS_rp_err","SDSS_ip_err","SDSS_zp_err",
]

# define h5 file column description
h5_columns = define_column_desc(
    magnitude_column_names=magnitude_column_names, 
    id_name_length=25, 
)

In [7]:
h5_columns

{'id_name': StringCol(itemsize=25, shape=(), dflt=np.bytes_(b''), pos=0),
 'ra': Float32Col(shape=(), dflt=np.float32(0.0), pos=1),
 'ra_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=2),
 'dec': Float32Col(shape=(), dflt=np.float32(0.0), pos=3),
 'dec_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=4),
 'SDSS_up': Float32Col(shape=(), dflt=np.float32(0.0), pos=5),
 'SDSS_gp': Float32Col(shape=(), dflt=np.float32(0.0), pos=6),
 'SDSS_rp': Float32Col(shape=(), dflt=np.float32(0.0), pos=7),
 'SDSS_ip': Float32Col(shape=(), dflt=np.float32(0.0), pos=8),
 'SDSS_zp': Float32Col(shape=(), dflt=np.float32(0.0), pos=9),
 'SDSS_up_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=10),
 'SDSS_gp_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=11),
 'SDSS_rp_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=12),
 'SDSS_ip_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=13),
 'SDSS_zp_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=14),
 'ipix': Int32Col(shape=(), d

# Read the cvs files

In [8]:
# file path
root_dir = Path("Northern_ugriz_processed")
files = list(root_dir.glob("*csv"))
files

[PosixPath('Northern_ugriz_processed/Hilt733_processed.csv'),
 PosixPath('Northern_ugriz_processed/Ross530_processed.csv'),
 PosixPath('Northern_ugriz_processed/Wolf1447_processed.csv'),
 PosixPath('Northern_ugriz_processed/97_d_processed.csv'),
 PosixPath('Northern_ugriz_processed/Hilt1089_processed.csv'),
 PosixPath('Northern_ugriz_processed/GD246_processed.csv'),
 PosixPath('Northern_ugriz_processed/Ross106_processed.csv'),
 PosixPath('Northern_ugriz_processed/Ross374_processed.csv'),
 PosixPath('Northern_ugriz_processed/BD+38o4955_processed.csv'),
 PosixPath('Northern_ugriz_processed/GCRV5951_processed.csv'),
 PosixPath('Northern_ugriz_processed/Ross49_processed.csv'),
 PosixPath('Northern_ugriz_processed/101_a_processed.csv'),
 PosixPath('Northern_ugriz_processed/BD+25o1981_processed.csv'),
 PosixPath('Northern_ugriz_processed/Hilt190_processed.csv'),
 PosixPath('Northern_ugriz_processed/95_f_processed.csv'),
 PosixPath('Northern_ugriz_processed/92_a_processed.csv'),
 PosixPath('N

In [9]:
# csv column names
colnames = [
    "id_name", "ra", "dec",
    "SDSS_up", "SDSS_gp","SDSS_rp", "SDSS_ip","SDSS_zp",
    "SDSS_up_err", "SDSS_gp_err", "SDSS_rp_err", "SDSS_ip_err", "SDSS_zp_err",
]

dataframe_iterator = iter_multi_csv_chunks(
    files=files, 
    chunksize=10, 
    read_csv_kwargs={
        "sep": ",", 
        "engine": "python", 
        "header":0,
    }
)

In [11]:
table_attrs = {
    "source": "https://www-star.fnal.gov/NorthEqExtension_ugriz/index.html",
    "ra_unit": "deg",
    "dec_unit": "deg",
    "ra_err_unit": "arcsec",
    "dec_err_unit": "arcsec",
    "version": "10 September 2012 (v3)",
    "mag_system": {
        "SDSS_ugriz": "AB",
    },
}

In [12]:
write_std_h5(
    dataframe_iterator=dataframe_iterator, 
    ra_dec_hmsdms=True,
    h5_output_path=Path("Smith_Northern_Sky.h5",), 
    group_where="/smith", 
    group_name="northern", 
    group_title="Northern Sky", 
    table_name="std", 
    table_description=h5_columns, 
    table_title="Standard Stars", 
    table_attrs=table_attrs, 
    nside=512, 
    bucket_size=1536 , 
)

Files:   0%|          | 0/110 [00:00<?, ?it/s]

Chunks:   0%|                                             | 0/3 [00:00<?, ?it/s]

Chunks:   0%|                                             | 0/2 [00:00<?, ?it/s]

Chunks:   0%|                                             | 0/1 [00:00<?, ?it/s]

Chunks:   0%|                                             | 0/2 [00:00<?, ?it/s]

...

Chunks:   0%|                                             | 0/8 [00:00<?, ?it/s]

Chunks:   0%|                                             | 0/2 [00:00<?, ?it/s]

Chunks:   0%|                                             | 0/1 [00:00<?, ?it/s]

Chunks:   0%|                                             | 0/1 [00:00<?, ?it/s]

Chunks:   0%|                                             | 0/5 [00:00<?, ?it/s]